In [ ]:
import pandas as pd  
import numpy as np
from tqdm import tqdm  
from collections import defaultdict  
import os, math, warnings, math, pickle
from tqdm import tqdm
import faiss
import collections
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model
# from tensorflow.python.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences
from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss
warnings.filterwarnings('ignore')

import gc

from deepmatch.models import YoutubeDNN
import tensorflow as tf
from tensorflow.keras.layers import *

DeepCTR version 0.9.3 detected. Your version is 0.8.2.
Use `pip install -U deepctr` to upgrade.Changelog: https://github.com/shenweichen/DeepCTR/releases/tag/v0.9.3


In [3]:
data_path = './data/'
save_path = './temp_results/'
# 做召回评估的一个标志, 如果不进行评估就是直接使用全量数据进行召回
metric_recall = False

In [4]:
# debug
def get_all_click_sample(data_path, sample_nums = 10000):
    all_click = pd.read_csv(data_path + 'train_click_log.csv')
    all_user_ids = all_click.user_id.unique()
    sample_user_ids = np.random.choice(all_user_ids, size = sample_nums, replace=False)
    all_click = all_click[all_click['user_id'].isin(sample_user_ids)]
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

def get_all_click_df(data_path = data_path, offline = True):
    if offline:
        all_click = pd.read_csv(data_path + 'train_click_log.csv')
    else:
        trn_click = pd.read_csv(data_path + 'train_click_log.csv')
        tst_click = pd.read_csv(data_path + 'testA_click_log.csv')
        all_click = trn_click.append(tst_click)
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click
    
def get_item_info_df(data_path):
    item_info_df = pd.read_csv(data_path + 'articles.csv')
    # 方便与训练集中的click_article_id拼接
    item_info_df = item_info_df.rename(columns={'article_id': 'click_article_id'})
    return item_info_df
def get_item_emb_dict(data_path):
    item_emb_df = pd.read_csv(data_path + 'articles_emb.csv')
    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols])
    # 归一化， 将每个向量变成长度=1
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)
    item_emb_dict = dict(zip(item_emb_df['article_id'], item_emb_np))
    pickle.dump(item_emb_dict, open(save_path + 'item_content_emb.pkl', 'wb'))
    return item_emb_dict

In [5]:
max_min_scaler = lambda x: (x-np.min(x))/(np.max(x) - np.min(x))

# 采样数据集
all_click_df = get_all_click_sample(data_path)
# 全量
# all_click_df = get_all_click_df(offline=False)
all_click_df['click_timestamp'] = all_click_df[['click_timestamp']].apply(max_min_scaler)

item_info_df = get_item_info_df(data_path)
item_emb_dict = get_item_emb_dict(data_path)

DeepMatch version 0.3.1 detected. Your version is 0.2.0.
Use `pip install -U deepmatch` to upgrade.Changelog: https://github.com/shenweichen/DeepMatch/releases/tag/v0.3.1


In [18]:
# 根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
def get_user_item_time(click_df):
    click_df = click_df.sort_values('click_timestamp')
    def make_item_time_pair(df):
        return list(zip(df['click_article_id'], df['click_timestamp']))
    user_item_time_df = click_df.groupby('user_id')[['click_article_id', 'click_timestamp']].apply(
        lambda x: make_item_time_pair(x)
    ).reset_index().rename(columns = {0: 'item_time_list'})
    user_item_time_dict = dict(zip(user_item_time_df['user_id'], user_item_time_df['item_time_list']))
    return user_item_time_dict

# 根据时间获取商品被点击的用户序列 {item1: {user1: time1, user2: time2...}...}
def get_item_user_time_dict(click_df):
    def make_user_time_pair(df):
        return list(zip(df['user_id'], df['click_timestamp']))
    click_df = click_df.sort_values('click_timestamp')
    item_user_time_df = click_df.groupby('click_article_id')[['user_id', 'click_timestamp']].apply(lambda x: make_user_time_pair(x)).reset_index().rename(columns={0: 'user_time_list'})
    item_user_time_dict = dict(zip(item_user_time_df['click_article_id'], item_user_time_df['user_time_list']))
    return item_user_time_dict

In [19]:
# 获取当前数据的历史点击、最后一次点击
def get_hist_and_last_click(all_click):
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)
    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]
    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)
    return click_hist_df, click_last_df

# 获取文章属性
def get_item_info_dict(item_info_df):
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts'] = item_info_df[['created_at_ts']].apply(max_min_scaler)
    
    item_type_dict = dict(zip(item_info_df['click_article_id'], item_info_df['category_id']))
    item_words_dict = dict(zip(item_info_df['click_article_id'], item_info_df['words_count']))
    item_created_time_dict = dict(zip(item_info_df['click_article_id'], item_info_df['created_at_ts']))
    
    return item_type_dict, item_words_dict, item_created_time_dict

# 获取用户历史点击的文章信息
def get_user_hist_item_info_dict(all_click):
    # 获取user_id对应的用户历史点击文章类型
    user_hist_item_typs = all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_typs_dict = dict(zip(user_hist_item_typs['user_id'], user_hist_item_typs['category_id']))
    # 获取user_id对应的用户点击文章集合
    user_hist_item_ids_dict = all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict = dict(zip(user_hist_item_ids_dict['user_id'], user_hist_item_ids_dict['click_article_id']))
    # 获取user_id对应的用户历史点击文字的平均数
    user_hist_item_words = all_click.groupby('user_id')['words_count'].agg('mean').reset_index()
    user_hist_item_words_dict = dict(zip(user_hist_item_words['user_id'], user_hist_item_words['words_count']))
    # 获取user_id对应的用户最后一次点击的文章的创建时间
    all_click_ = all_click.sort_values('click_timestamp')
    user_last_item_created_time = all_click_.groupby('user_id')['created_at_ts'].apply(lambda x:x.iloc[-1]).reset_index()

    max_min_scaler = lambda x: (x-np.min(x))/(np.max(x)- np.min(x))
    user_last_item_created_time['created_at_ts'] = user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    user_last_item_created_time_dict = dict(zip(user_last_item_created_time['user_id'], user_last_item_created_time['created_at_ts']))
    return user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict

# 近期点击次数最多的top-k文章
def get_item_topk_click(click_df, k):
    topk_click = click_df['click_article_id'].value_counts().index[:k]
    return topk_click


## 多路召回字典

In [20]:
# 文章属性信息
item_type_dict, item_words_dict, item_created_time_dict = get_item_info_dict(item_info_df)
# 多路召回字典，用于保存各路召回结果
user_multi_recall_dict = {
    'item_sim_itemcf_recall':{},
    'embedding_sim_item_recall':{},
    'youtubednn_recall':{},
    'youtubednn_usercf_recall':{},
    'cold_start_recall':{}
}

In [21]:
# 提取最后一次点击作为召回评估，如果不需要做召回评估直接使用全量的训练集进行召回(线下验证模型)
# 如果不是召回评估，直接使用全量数据进行召回，不用将最后一次提取出来
trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)

## 召回效果评估    
对当前的召回方法评估，为参数调整指方向

In [22]:
# 评估召回的前10， 20， 30， 40， 50个文章的击中率
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=5):
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    user_num = len(user_recall_items_dict)
    for k in range(10, topk+1, 10):
        hit_num = 0 #命中次数
        # 遍历每个用户的推荐列表
        for user, item_list in user_recall_items_dict.items():
            #用户前k个推荐文章
            tmp_recall_items = [x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp_recall_items):
                hit_num += 1
        # 求命中率
        hit_rate = round(hit_num * 1.0 / user_num, 5)
        print('topk:', k, ':', 'hit_num', hit_num, 'hit_rate', hit_rate, 'user_num', user_num)

## 求相似性矩阵

kdd 2020 去偏商品推荐
求相似度矩阵时还考虑到 用户点击的时间权重、用户点击顺序权重、文章创建时间权重

In [23]:
# itemcf
def itemcf_sim(df, item_created_time_dict):
    user_item_time_dict = get_user_item_time(df)
    # 求物品相似度
    i2i_sim = {}
    item_cnt = defaultdict(int)
    for user, item_time_list in tqdm(user_item_time_dict.items()):
        for loc1, (i, i_click_time) in enumerate(item_time_list):   # i为当前文章
            item_cnt[i] += 1   # 每个文章被多少用户点击过
            i2i_sim.setdefault(i, {})
            for loc2, (j, j_click_time) in enumerate(item_time_list): # j为同一用户点击过的另一个文章
                if i==j:
                    continue
                # 文章正向顺序点击和反向顺序点击
                loc_alpha = 1.0 if loc2>loc1 else 0.7
                # 位置信息权重
                loc_weight = loc_alpha * (0.9 ** (np.abs(loc2-loc1)-1))
                # 点击时间权重
                click_time_weight = np.exp(0.7 ** np.abs(i_click_time - j_click_time))
                # 两篇文章创建时间的权重
                created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                i2i_sim[i].setdefault(j,0)
                i2i_sim[i][j] += loc_weight * click_time_weight * created_time_weight / math.log(len(item_time_list) + 1)
    i2i_sim_ = i2i_sim.copy()
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij/math.sqrt(item_cnt[i] * item_cnt[j])
    pickle.dump(i2i_sim_, open(save_path + 'itemcf_i2i_sim.pkl', 'wb'))
    return i2i_sim_

In [24]:
i2i_sim = itemcf_sim(all_click_df, item_created_time_dict)

100%|██████████| 10000/10000 [00:02<00:00, 4662.51it/s]


简单的关联规则，如用户活跃度权重，这里将用户的点击次数作为用户活跃度的指标

In [25]:
# 得到用户活跃度
def get_user_activate_degree_dict(all_click_df):
    all_click_df_ = all_click_df.groupby('user_id')['click_article_id'].count().reset_index()
    
    # 用户活跃度归一化
    mm = MinMaxScaler()
    all_click_df_['click_article_id'] = mm.fit_transform(all_click_df_[['click_article_id']])
    user_activate_degree_dict = dict(zip(all_click_df_['user_id'], all_click_df_['click_article_id']))
    
    return user_activate_degree_dict

# usercf
def usercf_sim(all_click_df, user_activate_degree_dict):
    item_user_time_dict = get_item_user_time_dict(all_click_df)
    # 得到{item1: [(user1, t1), (user2, t2)],item2: [(user2, t3), (user3, t4)]}
    u2u_sim = {}  # u2u_sim[u][v]：用户u和v的相似度
    user_cnt = defaultdict(int) # user_cnt[u]：用户u点过多少文章
    for item, user_time_list in tqdm(item_user_time_dict.items()): #遍历文章item
        # 双循环，枚举点过同一篇文章的 (用户u, 用户v)
        for u, click_time  in user_time_list:
            user_cnt[u] += 1
            u2u_sim.setdefault(u, {})
            for v, click_time  in user_time_list:
                u2u_sim[u].setdefault(v, 0)
                if u == v:
                    continue
                # 相似度累加 用户平均活跃度作为活跃度的权重
                activate_weight = 100 * 0.5 * (user_activate_degree_dict[u] + user_activate_degree_dict[v])
                u2u_sim[u][v] += activate_weight / math.log(len(user_time_list) + 1)  # 分母：惩罚热门文章
    u2u_sim_ = u2u_sim.copy()   # {u1: {u2: 10, u3: 5}, u2: {u1: 10, u3: 2}, ...}   数字为加权共现次数
    for u, related_users in u2u_sim.items():
        for v, wij in related_users.items():
            # 归一化
            u2u_sim_[u][v] = wij / math.sqrt(user_cnt[u] * user_cnt[v])
            # 分母 惩罚过度活跃用户 相似度 = 共现次数 / sqrt(用户u活跃度 × 用户v活跃度)
    pickle.dump(u2u_sim_, open(save_path + 'usercf_u2u_sim.pkl', 'wb'))
    return u2u_sim_


user_activate_degree_dict = get_user_activate_degree_dict(all_click_df)
u2u_sim = usercf_sim(all_click_df, user_activate_degree_dict)

100%|██████████| 6472/6472 [00:01<00:00, 3298.27it/s]


In [17]:
print(len(user_activate_degree_dict))
print(all_click_df['user_id'].nunique())

10000
10000


## Youtube DNN

In [ ]:
# 负采样 获取训练数据
def gen_data_set(data, negsample = 0):
    data.sort_values('click_timestamp', inplace = True)
    item_ids = data['click_article_id'].unique()
    train_set = []
    test_set = []
    for reviewerID, hist in tqdm(data.groupby('user_id')): #挨个用户处理
        pos_list = hist['click_article_id'].tolist()  # 点过的所有文章转成列表
        if negsample > 0:
            # 生成负样本候选集
            candidate_set = list(set(item_ids) - set(pos_list))
            # 随机选负样本 负样本数量 = 正样本数量 * negsample
            neg_list = np.random.choice(candidate_set, size=len(pos_list)*negsample, replace=True)
        # 特殊情况：用户只点击了一篇文章
        if len(pos_list) == 1:
            # 此时必须放入训练集，否则用户/文章embedding无法学习到
            # 样本格式：(用户ID, 历史点击序列, 目标文章, 标签1=正, 历史长度)
            train_set.append(reviewerID, [pos_list[0]], pos_list[0], 1, len(pos_list))
            test_set.append(reviewerID, [pos_list[0]], pos_list[0], 1, len(pos_list))
        # 滑窗构造正负样本 用前 i 篇文章当历史，预测第 i+1 篇文章
        for i in range(1, len(pos_list)):
            hist = pos_list[:i] #当前历史为第i篇文章

            if i != len(pos_list)-1:    # i不是列表最后一个，放进训练集；否则放进测试集
                # 加入正样本 (用户ID, 倒序历史, 目标文章, 标签1, 历史长度)
                # hist[::-1]为历史倒序 让最近点击的文章排在最前面 [A, B, C] 变为[C, B, A]
                train_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))

                # 加负样本
                for negi in range(negsample):
                    train_set.append((reviewerID, hist[::-1], neg_list[i*negsample+negi], 0, len(hist[::-1])))
            else:
                # i是列表最后一个，只用完整最长历史做测试
                test_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set, test_set

# 将输入数据padding， 保证序列特征长度一致
# seq_max_len：历史序列最大长度
def gen_model_input(train_set,user_profile,seq_max_len):

    train_uid = np.array([line[0] for line in train_set])
    train_seq = [line[1] for line in train_set]
    train_iid = np.array([line[2] for line in train_set])
    train_label = np.array([line[3] for line in train_set])
    train_hist_len = np.array([line[4] for line in train_set])
    # 序列补齐  不够补0，超过截断
    train_seq_pad = pad_sequences(train_seq, maxlen=seq_max_len, padding='post', truncating='post', value=0)
    train_model_input = {"user_id": train_uid, "click_article_id": train_iid, "hist_article_id": train_seq_pad,
                         "hist_len": train_hist_len}

    return train_model_input, train_label


用 YouTubeDNN 模型训练用户和文章的向量（Embedding），然后用向量相似度给每个用户推荐最相似的 20 篇文章，最后保存结果、评估效果    

流程:    
把用户 ID、文章 ID 编码成数字    
生成训练样本（滑窗、历史序列→预测下一篇）    
搭建 YouTubeDNN 模型训练    
拿到 用户向量 和 文章向量    
用向量相似度搜索（faiss）给每个用户找最相似的文章    
保存向量 + 保存推荐结果    
评估召回效果

In [ ]:
# 输入点击日志，输出每个用户的top20推荐文章
def youtubednn_u2i_dict(data, topk=20):
    sparse_features = ['click_article_id', 'user_id']
    SEQ_LEN = 30
    user_profile_ = data[['user_id']].drop_duplicates('user_id')
    item_profile_ = data[['click_article_id']].drop_duplicates('click_article_id')

    # 类别编码
    features = ['click_article_id', 'user_id']
    feature_max_idx = {}
    for feature in features:
        lbe = LabelEncoder()
        data[feature] = lbe.fit_transform(data[feature])
        feature_max_idx[feature] = data[feature].max() + 1

    user_profile = data[["user_id"]].drop_duplicates('user_id')
    item_profile = data[["click_article_id"]].drop_duplicates('click_article_id')
    # 保存 “编码 ID ↔ 原始 ID” 的映射
    user_index_2_rawid = dict(zip(user_profile['user_id'], user_profile_['user_id']))
    item_index_2_rawid = dict(zip(item_profile['click_article_id'], item_profile_['click_article_id']))

    # 划分测试训练集
    train_set, test_set = gen_data_set(data, 0)
    train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
    test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

    embedding_dim = 16
    # 用户塔的输入特征
    user_feature_columns = [SparseFeat('user_id', feature_max_idx['user_id'], embedding_dim),
                            #给模型喂一个叫 user_id 的特征，此特征总共有 feature_max_idx['user_id'] 个不同的用户，每个用户要给生成 embedding_dim 维的向量
                            VarLenSparseFeat(SparseFeat('hist_article_id', feature_max_idx['click_article_id'], embedding_dim, embedding_name="click_article_id"),
                            # 历史点击的每一篇文章，都要学一个 16 维向量； 历史序列最长保留 30 篇
                            SEQ_LEN, 'mean', 'hist_len')   # 平均池化； histlen为真实长度，模型可知道哪些是0填充的
                            ]
            # user_feature_columns 有两大部分 [1. 给每个用户学一个专属向量, 2. 把用户历史看过的文章学向量再平均成兴趣向量]

    # 物品塔的输入特征
    item_feature_columns = [SparseFeat('click_article_id', feature_max_idx['click_article_id'], embedding_dim)] # 每篇文章学一个16维向量

    tf.compat.v1.disable_eager_execution()
    model = YoutubeDNN(user_feature_columns, item_feature_columns, num_sampled=5, user_dnn_hidden_units=(64, embedding_dim))
    # 模型编译
    model.compile(optimizer="adam", loss=sampledsoftmaxloss)  
    # 模型训练
    history = model.fit(train_model_input, train_label, batch_size=256, epochs=1, verbose=1, validation_split=0.0)

    # 提取训练的user item端embedding
    test_user_model_input = test_model_input
    all_item_model_input = {"click_article_id": item_profile['click_article_id'].values}
    user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
    item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
    # 保存当前embedding
    user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)
    item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)
    # embedding保存之前归一化一下
    user_embs = user_embs / np.linalg.norm(user_embs, axis=1, keepdims=True)
    item_embs = item_embs / np.linalg.norm(item_embs, axis=1, keepdims=True)
    # 将Embedding转换成字典
    raw_user_id_emb_dict = {user_index_2_rawid[k]: v for k, v in zip(user_profile['user_id'], user_embs)}
    raw_item_id_emb_dict = {item_index_2_rawid[k]: v for k, v in zip(item_profile['click_article_id'], item_embs)}
    # 将Embedding保存到本地
    pickle.dump(raw_user_id_emb_dict, open(save_path + 'user_youtube_emb.pkl', 'wb'))
    pickle.dump(raw_item_id_emb_dict, open(save_path + 'item_youtube_emb.pkl', 'wb'))

    # 向量搜索找与user_embedding相似性最高的topk item
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(item_embs) # 将所有item emb存入faiss
    sim, idx = index.search(np.ascontiguosarray(user_embs), topk)
    user_recall_items_dict = collections.defaultdict(dict)  # 存推荐结果 {用户原始ID: {文章原始ID: 相似度}}
    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(test_user_model_input['user_id'], sim, idx)):
        # target_idx当前遍历的用户编码; ID sim_value_list 这个用户对应的 20 个相似度分数; rele_idx_list 这个用户对应的 20 个文章编码索引
        target_raw_id = user_index_2_rawid[target_idx]
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]):  #从1开始是因为faiss检索的第一个结果永远是自己
            rele_raw_id = item_index_2_rawid[rele_idx]
            user_recall_items_dict[target_raw_id][rele_raw_id] = user_recall_items_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value
    # 推荐结果排序
    user_recall_items_dict = {k: sorted(v.items(), key=lambda x: x[1], reverse=True) for k, v in user_recall_items_dict.items()}

    pickle.dump(user_recall_items_dict, open(save_path + 'youtube_u2i_dict.pkl', 'wb'))
    return user_recall_items_dict




# 由于这里需要做召回评估，所以讲训练集中的最后一次点击都提取了出来
if not metric_recall: 
    # metric_recall=False → 不测试，直接生成推荐结果； =True → 模拟考试，测试模型准不准
    # 直接全量数据训练+推荐
    user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(all_click_df, topk=20)
else:
    # 做评估
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
    # 训练集训练模型
    user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(trn_hist_click_df, topk=20)
    # 测试集评估召回效果
    metrics_recall(user_multi_recall_dict['youtubednn_recall'], trn_last_click_df, topk=20)


100%|██████████| 10000/10000 [00:00<00:00, 28721.54it/s]


FailedPreconditionError: in user code:

    File "c:\Users\Administrator\Desktop\tianchi\deep_env\Lib\site-packages\deepctr\layers\core.py", line 193, in call  *
        fc = self.activation_layers[i](fc)
    File "c:\Users\Administrator\Desktop\tianchi\deep_env\Lib\site-packages\keras\src\engine\base_layer_v1.py", line 798, in __call__  **
        base_layer_utils.create_keras_history(inputs)
    File "c:\Users\Administrator\Desktop\tianchi\deep_env\Lib\site-packages\keras\src\engine\base_layer_utils.py", line 211, in create_keras_history
        _, created_layers = _create_keras_history_helper(tensors, set(), [])
    File "c:\Users\Administrator\Desktop\tianchi\deep_env\Lib\site-packages\keras\src\engine\base_layer_utils.py", line 295, in _create_keras_history_helper
        constants[i] = backend.function([], op_input)([])
    File "c:\Users\Administrator\Desktop\tianchi\deep_env\Lib\site-packages\keras\src\backend.py", line 4609, in __call__
        fetched = self._callable_fn(*arr

In [ ]:
# ===================== 你之前的两个工具函数（不动） =====================
def gen_data_set(data, negsample=0):
    data.sort_values("click_timestamp", inplace=True)
    item_ids = data['click_article_id'].unique()
    train_set = []
    test_set = []
    for reviewerID, hist in tqdm(data.groupby('user_id')):
        pos_list = hist['click_article_id'].tolist()
        if negsample > 0:
            candidate_set = list(set(item_ids) - set(pos_list))
            neg_list = np.random.choice(candidate_set,size=len(pos_list)*negsample,replace=True)
        if len(pos_list) == 1:
            train_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
            test_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
        for i in range(1, len(pos_list)):
            hist = pos_list[:i]
            if i != len(pos_list) - 1:
                train_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))
                for negi in range(negsample):
                    train_set.append((reviewerID, hist[::-1], neg_list[i*negsample+negi], 0,len(hist[::-1])))
            else:
                test_set.append((reviewerID, hist[::-1], pos_list[i],1,len(hist[::-1])))
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set, test_set

def gen_model_input(train_set,user_profile,seq_max_len):
    train_uid = np.array([line[0] for line in train_set])
    train_seq = [line[1] for line in train_set]
    train_iid = np.array([line[2] for line in train_set])
    train_label = np.array([line[3] for line in train_set])
    train_hist_len = np.array([line[4] for line in train_set])
    train_seq_pad = pad_sequences(train_seq, maxlen=seq_max_len, padding='post', truncating='post', value=0)
    train_model_input = {"user_id": train_uid, "click_article_id": train_iid, "hist_article_id": train_seq_pad, "hist_len": train_hist_len}
    return train_model_input, train_label

# ===================== 🔥 纯TF2.x重写的YouTubeDNN（替代deepmatch） =====================
def simple_youtube_dnn(user_vocab, item_vocab, seq_len, emb_dim=16):
    # 严格定义输入形状（修复报错核心）
    user_id = Input(shape=(1,), name="user_id", dtype=tf.int32)
    item_id = Input(shape=(1,), name="click_article_id", dtype=tf.int32)
    hist_item = Input(shape=(seq_len,), name="hist_article_id", dtype=tf.int32)
    hist_len = Input(shape=(1,), name="hist_len", dtype=tf.int32)

    # 用户Embedding + 展平（固定形状）
    user_emb = Embedding(user_vocab, emb_dim, name="user_emb")(user_id)
    user_emb_flat = Reshape((emb_dim,))(user_emb)  # (B, 16)

    # 物品Embedding共享
    item_emb_layer = Embedding(item_vocab, emb_dim, name="item_emb")
    item_emb = item_emb_layer(item_id)
    item_emb_flat = Reshape((emb_dim,))(item_emb)  # (B, 16)

    # 历史序列Embedding + 均值池化（绝对稳定）
    hist_emb = item_emb_layer(hist_item)  # (B, 30, 16)
    hist_mean = GlobalAveragePooling1D()(hist_emb)  # (B, 16)

    # 拼接：用户ID向量 + 历史兴趣向量（形状完全一致，无报错）
    user_concat = Concatenate(axis=-1)([user_emb_flat, hist_mean])

    # 用户塔DNN
    user_dnn = Dense(64, activation="relu", name="dnn_1")(user_concat)
    user_out = Dense(emb_dim, activation=None, name="user_out")(user_dnn)  # (B, 16)

    # 计算相似度（训练任务）
    dot_product = Multiply()([user_out, item_emb_flat])
    score = Lambda(lambda x: tf.reduce_sum(x, axis=1, keepdims=True))(dot_product)

    # 完整训练模型
    model = Model(inputs=[user_id, item_id, hist_item, hist_len], outputs=score)
    model.compile(optimizer="adam", loss="mse")

    # 抽取向量模型（推理用）
    user_model = Model(inputs=[user_id, hist_item, hist_len], outputs=user_out)
    item_model = Model(inputs=item_id, outputs=item_emb_flat)

    return model, user_model, item_model

# ===================== ✅ 主函数（100%兼容，无报错） =====================
def youtubednn_u2i_dict(data, topk=20):
    SEQ_LEN = 30
    user_profile_ = data[["user_id"]].drop_duplicates('user_id')
    item_profile_ = data[["click_article_id"]].drop_duplicates('click_article_id')  

    # ID编码
    features = ["click_article_id", "user_id"]
    feature_max_idx = {}
    for feature in features:
        lbe = LabelEncoder()
        data[feature] = lbe.fit_transform(data[feature])
        feature_max_idx[feature] = data[feature].max() + 1

    # 编码前后ID映射
    user_profile = data[["user_id"]].drop_duplicates('user_id')
    item_profile = data[["click_article_id"]].drop_duplicates('click_article_id')  
    user_index_2_rawid = dict(zip(user_profile['user_id'], user_profile_['user_id']))
    item_index_2_rawid = dict(zip(item_profile['click_article_id'], item_profile_['click_article_id']))

    # 数据构造
    train_set, test_set = gen_data_set(data, 0)
    train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
    test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

    embedding_dim = 16

    # 🔥 调用TF2.x原生模型（核心修复！）
    model, user_embedding_model, item_embedding_model = simple_youtube_dnn(
        user_vocab=feature_max_idx['user_id'],
        item_vocab=feature_max_idx['click_article_id'],
        seq_len=SEQ_LEN,
        emb_dim=embedding_dim
    )

    # 训练
    model.fit(train_model_input, train_label, batch_size=256, epochs=1, verbose=1)

    # 生成Embedding
    all_item_model_input = {"click_article_id": item_profile['click_article_id'].values}
    user_embs = user_embedding_model.predict(test_model_input, batch_size=4096, verbose=1)
    item_embs = item_embedding_model.predict(all_item_model_input, batch_size=4096, verbose=1)

    # 归一化
    user_embs = user_embs / np.linalg.norm(user_embs, axis=1, keepdims=True)
    item_embs = item_embs / np.linalg.norm(item_embs, axis=1, keepdims=True)

    # 保存Embedding
    save_path = './'
    raw_user_id_emb_dict = {user_index_2_rawid[k]:v for k,v in zip(user_profile['user_id'], user_embs)}
    raw_item_id_emb_dict = {item_index_2_rawid[k]:v for k,v in zip(item_profile['click_article_id'], item_embs)}
    pickle.dump(raw_user_id_emb_dict, open(save_path+'user_youtube_emb.pkl','wb'))
    pickle.dump(raw_item_id_emb_dict, open(save_path+'item_youtube_emb.pkl','wb'))

    # FAISS检索
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(item_embs)
    sim, idx = index.search(np.ascontiguousarray(user_embs), topk)

    # 组装结果
    user_recall_items_dict = collections.defaultdict(dict)
    for target_idx, sim_list, idx_list in tqdm(zip(test_model_input['user_id'], sim, idx)):
        uid = user_index_2_rawid[target_idx]
        for iid_idx, score in zip(idx_list[1:], sim_list[1:]):
            iid = item_index_2_rawid[iid_idx]
            user_recall_items_dict[uid][iid] = score
    user_recall_items_dict = {k:sorted(v.items(),key=lambda x:x[1],reverse=True) for k,v in user_recall_items_dict.items()}
    
    pickle.dump(user_recall_items_dict, open(save_path+'youtube_u2i_dict.pkl','wb'))
    return user_recall_items_dict

# ===================== 调用代码（原样不动） =====================
# 初始化字典
user_multi_recall_dict = {}
metric_recall = False  # 不评估，直接跑
topk=20

# 运行！
user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(all_click_df, topk=20)
print("运行完成！推荐结果已保存")

100%|██████████| 10000/10000 [00:00<00:00, 64286.14it/s]


Train on 35257 samples
35257/35257 [==============================] - 0s 8us/sample - loss: 0.4930


10000it [00:00, 434597.87it/s]


运行完成！推荐结果已保存
